In [1]:
import pandas as pd
import os
import glob
import shutil

In [2]:
# Here dict for names, for timestamp allignment and other utilities
NAS_PATH = os.path.expanduser("~/nas/EXO_RECORD")
SAVE_PATH = os.path.expanduser("~/nas/EXO_RECORD/raw_data_20260401/aligned")
ACTION = "action8"
TRIAL = "trial09"
TIME = {
    "camera_env" : 0, # First led timestamp
    "camera_body" : 1774982721130292736, # first led timestamp
    "frame_vicon_start": 564,
    "subframe_vicon_start" : 4, # Frames and subframes vicon. Sampling at 1000 Hz for convertion to frame_id delsys --> (df["Frame"] * VICON_SENSOR_HZ + df["Sub Frame"]*VICON_HZ)*1e9/VICON_SENSOR_HZ ho timestamp in ns
    "frame_vicon_end": 23008,
    "subframe_vicon_end" : 4,
}

CAMERA_BODY_HZ = 30
IMU_BODY_HZ = 200
VICON_HZ = 200
VICON_SENSOR_HZ = 1000
VICON_START = (TIME["frame_vicon_start"]/VICON_HZ + TIME["subframe_vicon_start"]/VICON_SENSOR_HZ)*1e9
VICON_INTERVAL = (TIME["frame_vicon_end"]/VICON_HZ + TIME["subframe_vicon_end"]/VICON_SENSOR_HZ)*1e9 - (TIME["frame_vicon_start"]/VICON_HZ + TIME["subframe_vicon_start"]/VICON_SENSOR_HZ)*1e9

if not os.path.exists(SAVE_PATH):
    os.makedirs(SAVE_PATH)

In [3]:
# Align body camera
camera_path = os.path.join(NAS_PATH, "raw_data_20260401/export_body_camera/action8")

# RGB saving
path_rgb = os.path.join(SAVE_PATH, "camera_body/rgb")
if not os.path.exists(path_rgb):
    os.makedirs(path_rgb)

all_rgb = (glob.glob(camera_path+"/camera_camera_color_image_raw_compressed/*.jpg"))
all_rgb.sort(key=lambda f: int(f.split('/')[-1].split('.')[0]))
rgb = [f for f in all_rgb if (int(f.split('/')[-1].split('.')[0]) >= TIME["camera_body"] and (TIME["camera_body"]+ VICON_INTERVAL + 1e9/CAMERA_BODY_HZ) > int(f.split('/')[-1].split('.')[0]))] # TODO: valutare se tenere anche il primo dopo
for i, f in enumerate(rgb):
        num = int(f.split('/')[-1].split('.')[0]) - TIME["camera_body"]
        final = os.path.join(path_rgb, f"{num}.jpg")
        shutil.copy2(f, final)
        
# Depth saving
path_depth = os.path.join(SAVE_PATH, "camera_body/depth")
if not os.path.exists(path_depth):
    os.makedirs(path_depth)

all_depth = (glob.glob(camera_path+"/camera_camera_aligned_depth_to_color_image_raw/*.png"))
all_depth.sort(key=lambda f: int(f.split('/')[-1].split('.')[0]))
depth = [f for f in all_depth if (int(f.split('/')[-1].split('.')[0]) >= TIME["camera_body"] and (TIME["camera_body"]+ VICON_INTERVAL + 1e9/CAMERA_BODY_HZ) >= int(f.split('/')[-1].split('.')[0]))]
for i, f in enumerate(depth):
        num = int(f.split('/')[-1].split('.')[0]) - TIME["camera_body"]
        final = os.path.join(path_depth, f"{num}.png")
        shutil.copy2(f, final)

In [17]:
# Saving camera IMU
imu_path = os.path.join(NAS_PATH, "raw_data_20260401/export_body_camera/action8/camera_camera_imu.csv")

df = pd.read_csv(imu_path)

df["timestamp"] = pd.to_numeric(df["timestamp"])*1e9

mask = ((df["timestamp"]-TIME["camera_body"])>(-1e9/IMU_BODY_HZ)) & ((VICON_INTERVAL+ 1e9/IMU_BODY_HZ)>(df["timestamp"]-TIME["camera_body"]))
df = df[mask]
start = df["timestamp"].iloc[0]
df["timestamp"] = df["timestamp"] - start
cols = ["timestamp", "angular_velocity_x", "angular_velocity_y", "angular_velocity_z", "linear_acceleration_x", "linear_acceleration_y", "linear_acceleration_z"]
df = df[cols]
df.rename(columns={"angular_velocity_x": "gyro_x", "angular_velocity_y": "gyro_y", "angular_velocity_z": "gyro_z", "linear_acceleration_x": "accel_x", "linear_acceleration_y": "accel_y", "linear_acceleration_z": "accel_z"}, inplace=True)

path_imu = SAVE_PATH
if not os.path.exists(path_imu):
    os.makedirs(path_imu)

df.to_csv(os.path.join(path_imu, "imu_camera.csv"), index=False)


In [6]:
# Parsing and saving Vicon sensors
sensor_path = os.path.join(NAS_PATH, "raw_data_20260401/Pilota/trial09.csv")

df = pd.read_csv(sensor_path, sep="\t", skiprows=3, header=0, encoding="utf-8-sig", low_memory=False)
df = df.iloc[1:].reset_index(drop=True)

df["Frame"] = pd.to_numeric(df["Frame"])
df["Sub Frame"] = pd.to_numeric(df["Sub Frame"])


key = (df["Frame"] * 5 + df["Sub Frame"])
key_start = (TIME["frame_vicon_start"]*5 + TIME["subframe_vicon_start"])
key_end   = (TIME["frame_vicon_end"]*5 + TIME["subframe_vicon_end"])
start_time = (TIME["frame_vicon_start"]/VICON_HZ + TIME["subframe_vicon_start"]/VICON_SENSOR_HZ)*1e9


df = df[(key >= key_start) & (key <= key_end)]

# Saving ZeroWires EMG
col_start = "DP_sx"
col_end   = "16"
cols_all  = list(df.columns)
idx_start = cols_all.index(col_start)
idx_end   = cols_all.index(col_end)

df_filt = df[cols_all[idx_start : idx_end + 1]].reset_index(drop=True)
df_filt['timestamp'] = (df["Frame"]/VICON_HZ + df["Sub Frame"]/VICON_SENSOR_HZ)*1e9 - VICON_START

df_filt.to_csv(os.path.join(SAVE_PATH, "emg_zerowire.csv"), index=False)

# Saving Force plate 
col_start = "Fx"
col_end   = "Cz"
cols_all  = list(df.columns)
idx_start = cols_all.index(col_start)
idx_end   = cols_all.index(col_end)

df_filt = df[cols_all[idx_start : idx_end + 1]].reset_index(drop=True)
df_filt['timestamp'] = (df["Frame"]/VICON_HZ + df["Sub Frame"]/VICON_SENSOR_HZ)*1e9 - VICON_START

df_filt.to_csv(os.path.join(SAVE_PATH, "forceplate.csv"), index=False)


In [7]:
# Parsing and saving the Delsys data

delsys_path = os.path.join(NAS_PATH, "raw_data_20260401/export_delsys/action8_delsys.csv")

def find_header_row(filepath, encoding="latin-1", keyword="X [s]"):
    """Find header leveraging on keyword"""
    with open(filepath, encoding=encoding, errors="replace") as f:
        for i, line in enumerate(f):
            if keyword in line:
                return i
    raise ValueError(f"no key '{keyword}' found.")

header_row = find_header_row(delsys_path)

df = pd.read_csv(delsys_path, sep="\t", skiprows=header_row, header=0, encoding="latin-1", low_memory=False)

emg_cols = []
acc_gyro_cols = []

cols = list(df.columns)
i = 0
while i < len(cols):
    col = cols[i]
    # Find X [s], next column is data column
    if col.startswith("X [s]"):
        if i + 1 < len(cols):
            next_col = cols[i + 1]
            if ": EMG" in next_col:
                emg_cols += [col, next_col]
            else:  # ACC o GYRO
                acc_gyro_cols += [col, next_col]
        i += 2
    else:
        i += 1

def filter_and_save(df, signal_cols, t_start, t_end, out_path):
    # use first X [s], all the others are the same
    ts_col    = signal_cols[0]
    data_cols = [c for c in signal_cols if not c.startswith("X [s]")]

    ts = pd.to_numeric(df[ts_col], errors="coerce")
    mask = (ts >= t_start) & (ts <= t_end)

    df_out = df.loc[mask, [ts_col] + data_cols].copy()
    df_out[ts_col] = df_out[ts_col] - df_out[ts_col].iloc[0]
    df_out = df_out.rename(columns={ts_col: "timestamp"})
    df_out["timestamp"]= df_out["timestamp"]
    df_out.to_csv(out_path, sep=",", index=False)

filter_and_save(df, emg_cols, (VICON_START)/1e9, (VICON_START+VICON_INTERVAL)/1e9, os.path.join(SAVE_PATH, "emg_delsys.csv"))
filter_and_save(df, acc_gyro_cols, (VICON_START)/1e9, (VICON_START+VICON_INTERVAL)/1e9, os.path.join(SAVE_PATH, "imu_delsys.csv"))

In [16]:
# Parsing and saving the VICON data (pre-PIG)
vicon_path = os.path.join(NAS_PATH, "raw_data_20260401/export_vicon/action8/trial09_pre_PIG.csv")

def find_header_row(filepath, encoding="utf-8", keyword="Trajectories"):
    """Find header leveraging on keyword"""
    with open(filepath, encoding=encoding, errors="replace") as f:
        for i, line in enumerate(f):
            if keyword in line:
                return i
    raise ValueError(f"no key '{keyword}' found.")

header_row = find_header_row(vicon_path)

raw = pd.read_csv(vicon_path, sep=",", skiprows=(header_row+2), header=None, encoding="utf-8", low_memory=False)

target_names = raw.iloc[0].fillna("").tolist()
axes         = raw.iloc[1].tolist()

col_names = []
current_target = ""
for t, ax in zip(target_names, axes):
    if str(t).strip():
        current_target = str(t).strip()
    ax = str(ax).strip()
    if ax in ("X", "Y", "Z"):
        col_names.append(f"{current_target}_{ax}")
    else:
        col_names.append(ax)

df = pd.read_csv(vicon_path, header=None, skiprows=(header_row+4))
df.columns = col_names[:len(df.columns)]

df = df[(df["Frame"] >= (TIME["frame_vicon_start"]+1)) & (df["Frame"] <= (TIME["frame_vicon_end"]+1))].copy()

df["timestamp"] = df["Frame"]*1e9 / VICON_HZ 

# Shift: primo timestamp a 0
df["timestamp"] -= df["timestamp"].iloc[0]


# Riordina: timestamp prima, poi rimuovi Frame e Sub Frame
data_cols = [c for c in df.columns if c not in ("Frame", "Sub Frame", "timestamp")]
df = df[["timestamp"] + data_cols]

df.to_csv(os.path.join(SAVE_PATH, "vicon.csv"), index=False)

In [ ]:
# Parsing and saving VICON data (all post-PIG)